## Making my own GPT

In [ ]:


from datasets import load_dataset
data = load_dataset("roneneldan/TinyStories",)

In [ ]:
# Tokenize the prompt
from transformers import AutoModel,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token # gt2 doesnt have padiing token makin that as eos token.
gpt_2 = AutoModel.from_pretrained("gpt2")

tokens = tokenizer(data["train"]['text'][:100],return_tensors="pt",padding=True,truncation=True) #tokens form user prompt as per gpt-2

Vector_embeddings = gpt_2.get_input_embeddings()

embeddings = Vector_embeddings(tokens["input_ids"]) # embeddings from the gpt-2 model


In [ ]:
# Tokens and the vector representation of it
print(f"Glimpse of tokens of the data :- {tokens["input_ids"][0][0:10]}")
print(f"length tokens of the data :- {len(tokens["input_ids"][0])}")

print("  ")
print("-------------------------------------")
print("  ")

print(f" shape of the Embeddings of the data :- {embeddings.shape}")
print(f" glimpse of the embeddigs :- {embeddings[0][0][0:5]}")

In [ ]:
# add positional encoding to the Vector Embeddings
import torch
import numpy as np
def positional_encoding(length_of_ids,embeddings_lengths):
    # this takes the same shape as vector embeddings gives u positional embeddings for it

    pos = length_of_ids
    d = embeddings_lengths
    pe = torch.zeros((1,pos,d))
    for i in range(pos):
        for j in range(d//2):
            pe[0,i,2*j] = np.sin(i / 10000 ** (2 * j / d ))
            pe[0,i,2*j+1] = np.cos(i / 10000 ** (2 * j/ d ))
    return pe


In [ ]:
#positional encodings
pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])


In [ ]:
embeddings.shape

In [ ]:
pe.shape

In [ ]:
# making pe ([200,298,768])

pe = pe.repeat(100,1,1)
pe.shape

In [ ]:
# final input embeddings = vector_embedding + positional encoding
input_embeddings = embeddings + pe

In [ ]:

print(f" shape of the input embeddings of the user prompt :- {input_embeddings.shape}")
print(f" glimpse of the input embeddigs :- {embeddings[0][0][0:5]}")

## Pre Attention Done

In [ ]:
input_embeddings = input_embeddings

residual_1 = input_embeddings.clone()

(input_embeddings@w_q).shape

In [ ]:


#init the wieghts

w_k = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)*0.02
w_v = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)*0.02
w_q = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1],requires_grad=True)*0.02

# calculating the Query , Key ,Value

Q = input_embeddings@w_q
K = input_embeddings@w_k
V = input_embeddings@w_v

num_head = 2
if 768%num_head ==0:
    dim = int(768/num_head)
else:
    raise "nigg chose diff num_head"

Q = Q.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)
K = K.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)
V = V.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)

# Attention Score
scores = (Q@K.transpose(-2,-1))/np.sqrt(dim)

mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
scores = scores.masked_fill(mask, float("-inf"))

Attention_Score = torch.nn.functional.softmax(scores,dim=-1)@V
Attention_out = Attention_Score.reshape(input_embeddings.shape) + residual_1
Attention_out_rms = torch.nn.functional.rms_norm(Attention_out,(Attention_out.shape[-1],))
Attention_rms = Attention_out_rms



In [ ]:
print(f"The shape of attention out {Attention_rms.shape}")
print(f"The shape of attention out {Attention_rms[0][:5]}")

## MultiHead Attention Block Done

In [ ]:
residual_2 = Attention_rms.clone()
from collections import OrderedDict
d_1=Attention_rms.shape[-2]
d_2=Attention_rms.shape[-1]

model = torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2)),
            ("relu1", torch.nn.ReLU()),
            ("Linear2", torch.nn.Linear(d_2,d_2*4)),
            ("relu2", torch.nn.ReLU()),
            ("Linear3", torch.nn.Linear(d_2*4,d_2*4)),
            ("relu3", torch.nn.ReLU()),
            ("Linear4", torch.nn.Linear(d_2*4,d_2))
        ]
    )
)
ffn_out = model(Attention_rms)
output = ffn_out + residual_2
output = torch.nn.functional.rms_norm(output,(768,))



In [ ]:

output.shape

## The FFn Layer Done


In [ ]:
layer = []
for i in range(10):
    layer.append({
        "w_q" : torch.rand(768,768,requires_grad=True)*0.02,
        "w_k" : torch.rand(768,768,requires_grad=True)*0.02,
        "w_v" : torch.rand(768,768,requires_grad=True)*0.02,
        "ffn" : torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2)),
            ("relu1", torch.nn.ReLU()),
            ("Linear2", torch.nn.Linear(d_2,d_2*4)),
            ("relu2", torch.nn.ReLU()),
            ("Linear3", torch.nn.Linear(d_2*4,d_2*4)),
            ("relu3", torch.nn.ReLU()),
            ("Linear4", torch.nn.Linear(d_2*4,d_2))
        ]
    )
)


    })


for i in range(10):
    r1 = output.clone()

    Q = output @ layer[i]["w_q"]
    K = output @ layer[i]["w_k"]
    V = output @ layer[i]["w_v"]



    Q = Q.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    K = K.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    V = V.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    scores = (Q @ K.transpose(-2,-1)) / np.sqrt(dim)
    mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
    scores = scores.masked_fill(mask, float("-inf"))
    Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
    Attn_out = Attention_score.reshape(output.shape)  + r1
    Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

    r2 = Attn_rms.clone()

    ffn_out = layer[i]["ffn"](Attn_rms)

    ffn_out = ffn_out + r2

    ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
    output = ffn_out





all_params = []

for i in layer:
    all_params += [i["w_k"],i["w_v"],i["w_q"]]
    all_params += list(i["ffn"].parameters())

all_params += list(model.parameters())
all_params += [w_k,w_q,w_v]




In [ ]:
output.shape

In [ ]:
vocab = 50257

output_projection = torch.nn.Linear(768,vocab)

logits = output_projection(output)

all_params += list(output_projection.parameters())

ids = tokens["input_ids"]
target = torch.cat([ids[1:],ids[-1:]],dim=0)

optimizer = torch.optim.Adam(all_params,lr=1e-2)
loss = torch.nn.functional.cross_entropy(logits,target)




In [ ]:
losses = []
for j in range(100):
    embeddings = Vector_embeddings(tokens["input_ids"])
    pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])
    input_embeddings = embeddings + pe.repeat(100,1,1)
    input_embeddings = input_embeddings
    output = input_embeddings

    residual_1 = input_embeddings.clone()
    Q = input_embeddings @ w_q
    K = input_embeddings @ w_k
    V = input_embeddings @ w_v
    Q = Q.reshape(output.shape[0],output.shape[1], 12, 64).transpose(-2, 1)
    K = K.reshape(output.shape[0],output.shape[1], 12, 64).transpose(-2, 1)
    V = V.reshape(output.shape[0],output.shape[1], 12, 64).transpose(-2, 1)
    scores = (Q @ K.transpose(-2,-1)) / np.sqrt(64)
    mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
    scores = scores.masked_fill(mask, float("-inf"))
    Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
    output = torch.nn.functional.rms_norm(Attention_score + residual_1, (768,))

    # train
    for i in range(10):

        r1 = output.clone()

        Q = output @ layer[i]["w_q"]
        K = output @ layer[i]["w_k"]
        V = output @ layer[i]["w_v"]

        Q = Q.reshape(output.shape[0],output.shape[1],12,64).transpose(-2,1)
        K = K.reshape(output.shape[0],output.shape[1],12,64).transpose(-2,1)
        V = V.reshape(output.shape[0],output.shape[1],12,64).transpose(-2,1)


        scores = (Q @ K.transpose(-2,-1)) / np.sqrt(64)
        mask = torch.triu(torch.ones(output.shape[0], output.shape[0]), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
        Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
        Attention_score = Attention_score.permute(1,0,2).reshape(output.shape[0],768)
        Attn_out = Attention_score  + r1
        Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

        r2 = Attn_rms.clone()

        ffn_out = layer[i]["ffn"](Attn_rms)

        ffn_out = ffn_out + r2

        ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
        output = ffn_out

    # calc loss backprop and update weights
    logits = output_projection(output)
    loss = torch.nn.functional.cross_entropy(logits,target)

    optimizer.zero_grad()
    torch.autograd.set_detect_anomaly(True, check_nan=False)
    loss.backward(retain_graph=True,)
    optimizer.step()

    losses.append(loss)

    if j%10 == 0:
        print(f"The loss at iteration {j} is {loss}")


In [ ]:
output.shape

In [ ]:
def generate(prompt,max_tokens = 20,temp = 1.0):
    tokens = tokenizer(prompt,return_tensors="pt")
    ids = tokens["input_ids"].squeeze(0)

    for _ in range(max_tokens):
        embeddings = Vector_embeddings(ids)
        pe = positional_encoding(embeddings.shape[0],embeddings.shape[1])
        input_embeddings = embeddings + pe
        input_embeddings = input_embeddings.squeeze(0)
        output = input_embeddings

        residual_1 = input_embeddings.clone()
        Q = input_embeddings @ w_q
        K = input_embeddings @ w_k
        V = input_embeddings @ w_v
        seq_len = input_embeddings.shape[0]
        Q = Q.reshape(seq_len, 12, 64).transpose(1, 0)
        K = K.reshape(seq_len, 12, 64).transpose(1, 0)
        V = V.reshape(seq_len, 12, 64).transpose(1, 0)
        attn = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64), dim=-1) @ V
        attn = attn.permute(1, 0, 2).reshape(seq_len, 768)
        output = torch.nn.functional.rms_norm(attn + residual_1, (768,))

        # train
        for i in range(12):

            r1 = output.clone()

            Q = output @ layer[i]["w_q"]
            K = output @ layer[i]["w_k"]
            V = output @ layer[i]["w_v"]

            Q = Q.reshape(output.shape[0],12,64).transpose(1,0)
            K = K.reshape(output.shape[0],12,64).transpose(1,0)
            V = V.reshape(output.shape[0],12,64).transpose(1,0)


            Attention_score = torch.nn.functional.softmax((Q @ K.transpose(-2,-1)) / np.sqrt(64) ,dim=-1)@V
            Attention_score = Attention_score.permute(1,0,2).reshape(output.shape[0],768)
            Attn_out = Attention_score  + r1
            Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))

            r2 = Attn_rms.clone()

            ffn_out = layer[i]["ffn"](Attn_rms)

            ffn_out = ffn_out + r2

            ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
            output = ffn_out



        logits = output_projection(output[-1])

        logits = logits/temp

        probs = torch.nn.functional.softmax(logits,dim=-1)

        next_token = torch.multinomial(probs,num_samples=1)

        ids = torch.cat([ids,next_token],dim=0)

        if next_token == tokenizer.eos_token_id:
                  break


    return tokenizer.decode(ids,skip_special_tokens=True)



In [ ]:
prompt = input("Enter the prompt : - ")

ans = generate(prompt,max_tokens=20,temp=1.0)

In [ ]:
ans